In [1]:
%config Completer.use_jedi = False
#Задание 3

In [2]:
import pandas as pd
import numpy as np

In [4]:
train_data = pd.read_csv("train.csv")

In [5]:
train_data

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


In [6]:
test_data = pd.read_csv("test.csv")

In [7]:
test_data

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...
413,1305,3,"Spector, Mr. Woolf",male,NaN,0,0,A.5. 3236,8.0500,NaN,S
414,1306,1,"Oliva y Ocana, Dona. Fermina",female,39.0,0,0,PC 17758,108.9000,C105,C
415,1307,3,"Saether, Mr. Simon Sivertsen",male,38.5,0,0,SOTON/O.Q. 3101262,7.2500,NaN,S
416,1308,3,"Ware, Mr. Frederick",male,NaN,0,0,359309,8.0500,NaN,S


In [31]:
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer
from sklearn.preprocessing import OneHotEncoder

In [32]:
def select_cols(mas):
    return mas[["Age", "SibSp", "Parch", "Fare"]]
def select_cols2(mas):
    return mas[["Pclass", "Sex", "Embarked"]]  

In [33]:
num_pipeline = Pipeline([
    ("select_nums", FunctionTransformer(select_cols, validate=False)),
    ("imputer", SimpleImputer(strategy="median")),
])

In [34]:
num_pipeline.fit_transform(train_data)

array([[22.    ,  1.    ,  0.    ,  7.25  ],
       [38.    ,  1.    ,  0.    , 71.2833],
       [26.    ,  0.    ,  0.    ,  7.925 ],
       ...,
       [28.    ,  1.    ,  2.    , 23.45  ],
       [26.    ,  0.    ,  0.    , 30.    ],
       [32.    ,  0.    ,  0.    ,  7.75  ]], shape=(891, 4))

In [35]:
class MostFrequentImputer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.most_frequent_ = pd.Series([X[c].value_counts().index[0] for c in X],
                                        index=X.columns)
        return self
    def transform(self, X, y=None):
        return X.fillna(self.most_frequent_)

In [37]:
cat_pipeline = Pipeline([
        ("select_cat", FunctionTransformer(select_cols2, validate=False)),
        ("imputer", MostFrequentImputer()),
        ("cat_encoder", OneHotEncoder()),
])

In [38]:
cat_pipeline.fit_transform(train_data)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 2673 stored elements and shape (891, 8)>

In [39]:
from sklearn.pipeline import FeatureUnion
preprocess_pipeline = FeatureUnion(transformer_list=[
        ("num_pipeline", num_pipeline),
        ("cat_pipeline", cat_pipeline),
    ])

In [40]:
X_train = preprocess_pipeline.fit_transform(train_data)
X_train

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4936 stored elements and shape (891, 12)>

In [41]:
y_train = train_data["Survived"]

In [42]:
y_train

0      0
1      1
2      1
3      1
4      0
      ..
886    0
887    1
888    0
889    1
890    0
Name: Survived, Length: 891, dtype: int64

In [43]:
X_test = preprocess_pipeline.fit_transform(test_data)
X_test

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 2317 stored elements and shape (418, 12)>

In [117]:
from sklearn.model_selection import GridSearchCV
# Метод опорных векторов
from sklearn.svm import SVC
svm_clf = SVC(gamma="auto")
svm_clf.fit(X_train, y_train)

SVC(gamma='auto')

In [118]:
y_predictOfSVC = svm_clf.predict(X_test)

In [119]:
from sklearn.model_selection import cross_val_score

svm_scores = cross_val_score(svm_clf, X_train, y_train, cv=16)
svm_scores.mean()

np.float64(0.7409699675324675)

In [120]:
# Логистическая регрессия
from sklearn.linear_model import LogisticRegression

In [121]:
logistic_reg = LogisticRegression(max_iter = 1000)
logistic_reg.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

In [122]:
y_predictOfLogReg = logistic_reg.predict(X_test)

In [123]:
LogReg_scores = cross_val_score(logistic_reg, X_train, y_train, cv=24)
LogReg_scores.mean()

np.float64(0.8036391654812708)

In [124]:
# Деревья решений
from sklearn.tree import DecisionTreeClassifier
tree_model = DecisionTreeClassifier()

In [125]:
tree_model.fit(X_train, y_train)

DecisionTreeClassifier()

In [126]:
y_predictOfDecTree = tree_model.predict(X_test)

In [127]:
DecTree_scores = cross_val_score(tree_model, X_train, y_train, cv=24)
DecTree_scores.mean()

np.float64(0.7935040303461355)

In [133]:
# Случайные леса
from sklearn.ensemble import RandomForestClassifier
forest_model = RandomForestClassifier(max_depth=None, min_samples_split=10, n_estimators=50)

In [134]:
forest_model.fit(X_train, y_train)

RandomForestClassifier(min_samples_split=10, n_estimators=50)

In [135]:
y_predictOfRandForest = forest_model.predict(X_test)

In [139]:
RandForest_scores = cross_val_score(forest_model, X_train, y_train, cv=25)
RandForest_scores.mean()

np.float64(0.8283809523809523)

In [140]:
# Градиентный бустинг
from sklearn.ensemble import GradientBoostingClassifier
gb_model = GradientBoostingClassifier()

In [141]:
gb_model.fit(X_train, y_train)

GradientBoostingClassifier()

In [142]:
y_predictOfGB = gb_model.predict(X_test)

In [144]:
GB_scores = cross_val_score(gb_model, X_train, y_train, cv=24)
GB_scores.mean()

np.float64(0.8272285443338075)

In [156]:
# K-ближайших соседей
from sklearn.neighbors import KNeighborsClassifier
knn_model = KNeighborsClassifier(metric='manhattan', n_neighbors=17, weights='distance')

In [157]:
knn_model.fit(X_train, y_train)

KNeighborsClassifier(metric='manhattan', n_neighbors=17, weights='distance')

In [158]:
y_predictOfKNN = knn_model.predict(X_test)

In [159]:
knn_scores = cross_val_score(knn_model, X_train, y_train, cv=16)
knn_scores.mean()

np.float64(0.7688717532467533)